# Expected Order Value Regression

This notebook compares 16 regression models, selects the model with the lowest validation RMSE, evaluates it on untouched test data, and saves it for the application.

## 1. Setup

In [1]:
from pathlib import Path
from collections import defaultdict
import csv

import joblib
import numpy as np
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    AdaBoostRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, RandomForestRegressor,
)
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import ElasticNet, HuberRegressor, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR, SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

### Locate the project and define paths

In [2]:
def find_project_root(start_directory):
    start_directory = Path(start_directory).resolve()
    for folder in [start_directory, *start_directory.parents]:
        if (folder / "data" / "user_order_history.csv").is_file():
            return folder
        if (folder / "culinary_matchmaker" / "data" / "user_order_history.csv").is_file():
            return folder / "culinary_matchmaker"
    raise FileNotFoundError("Could not find culinary_matchmaker/data/user_order_history.csv")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "user_order_history.csv"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

## 2. Build Historical Spending Features

Each row uses only the user's previous orders. The current order amount is added to the history after its feature row is created, preventing target leakage.

In [3]:
def engineer_spending_features(data_path):
    user_history = defaultdict(lambda: {"orders": 0, "amount_sum": 0.0})
    features, targets = [], []

    with data_path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        required_columns = {
            "user_id", "average_spend", "meal_time", "day_type",
            "location", "payment_method", "preferred_cuisine", "order_amount",
        }
        missing_columns = required_columns - set(reader.fieldnames or [])
        if missing_columns:
            raise ValueError(f"Missing columns: {sorted(missing_columns)}")

        for row in reader:
            history = user_history[row["user_id"]]
            previous_average = (
                history["amount_sum"] / history["orders"]
                if history["orders"] > 0
                else float(row["average_spend"])
            )

            features.append({
                "user_average_order_value": round(previous_average, 3),
                "meal_time": row["meal_time"].strip(),
                "weekday_or_weekend": row["day_type"].strip(),
                "location": row["location"].strip(),
                "payment_method": row["payment_method"].strip(),
                "previous_order_count": history["orders"],
                "preferred_cuisine": row["preferred_cuisine"].strip(),
            })

            order_amount = float(row["order_amount"])
            if order_amount < 0:
                raise ValueError("Order amount cannot be negative")
            targets.append(order_amount)
            history["orders"] += 1
            history["amount_sum"] += order_amount

    return features, targets

### Exclude users with no previous order

In [4]:
all_features, all_targets = engineer_spending_features(DATA_PATH)

returning_users = [
    (feature, target)
    for feature, target in zip(all_features, all_targets)
    if feature["previous_order_count"] >= 1
]
features = [feature for feature, _ in returning_users]
targets = np.array([target for _, target in returning_users])

print("Training records available:", len(features))

Training records available: 35000


## 3. Prepare Training, Validation, and Test Data

In [5]:
x_development_rows, x_test_rows, y_development, y_test = train_test_split(
    features, targets, test_size=0.15, random_state=RANDOM_STATE
)
x_train_rows, x_validation_rows, y_train, y_validation = train_test_split(
    x_development_rows, y_development, test_size=0.1764705882,
    random_state=RANDOM_STATE,
)

print("Train:", len(y_train), "Validation:", len(y_validation), "Test:", len(y_test))

Train: 24500 Validation: 5250 Test: 5250


### Encode categorical features and scale all values

In [6]:
preprocessor = Pipeline([
    ("vectorizer", DictVectorizer(sparse=False)),
    ("scaler", StandardScaler()),
])

x_train = preprocessor.fit_transform(x_train_rows).astype(np.float32)
x_validation = preprocessor.transform(x_validation_rows).astype(np.float32)

## 4. Compare Regression Models

In [7]:
models = {
    "Dummy Mean": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.10, max_iter=5000, random_state=RANDOM_STATE),
    "Elastic Net": ElasticNet(alpha=0.05, l1_ratio=0.35, max_iter=5000, random_state=RANDOM_STATE),
    "Huber Regression": HuberRegressor(epsilon=1.35, max_iter=1000),
    "Linear SVR": LinearSVR(C=1.0, epsilon=0.1, max_iter=20000, random_state=RANDOM_STATE),
    "RBF SVR": SVR(C=100.0, epsilon=5.0, gamma="scale", cache_size=700),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=2),
    "Decision Tree": DecisionTreeRegressor(max_depth=18, min_samples_leaf=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(
        n_estimators=220, max_depth=22, min_samples_leaf=3,
        max_features=0.75, n_jobs=2, random_state=RANDOM_STATE,
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=220, max_depth=24, min_samples_leaf=3,
        max_features=0.85, n_jobs=2, random_state=RANDOM_STATE,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=180, learning_rate=0.05, max_depth=3,
        loss="huber", random_state=RANDOM_STATE,
    ),
    "Histogram Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=220, learning_rate=0.06, max_leaf_nodes=31,
        l2_regularization=0.2, random_state=RANDOM_STATE,
    ),
    "AdaBoost": AdaBoostRegressor(
        n_estimators=160, learning_rate=0.05, loss="square", random_state=RANDOM_STATE
    ),
    "XGBoost": XGBRegressor(
        objective="reg:squarederror", n_estimators=300, max_depth=6,
        learning_rate=0.05, subsample=0.9, colsample_bytree=0.85,
        tree_method="hist", n_jobs=2, random_state=RANDOM_STATE,
    ),
}

### Define the evaluation metrics

In [8]:
def calculate_metrics(actual, predicted):
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": np.sqrt(mean_squared_error(actual, predicted)),
        "r2": r2_score(actual, predicted),
    }

### Limit only the slowest models

Huber Regression and RBF SVR use reproducible random samples during comparison because they are much slower on the complete dataset.

In [9]:
def get_training_data(model_name):
    training_limit = {"Huber Regression": 12000, "RBF SVR": 8000}.get(model_name)

    if training_limit is None or training_limit >= len(y_train):
        return x_train, y_train

    random_generator = np.random.default_rng(RANDOM_STATE)
    selected_rows = random_generator.choice(len(y_train), training_limit, replace=False)
    return x_train[selected_rows], y_train[selected_rows]

### Train every model and select the lowest validation RMSE

In [10]:
results = []

for name, model in models.items():
    model_x_train, model_y_train = get_training_data(name)
    trained_model = clone(model).fit(model_x_train, model_y_train)
    predictions = trained_model.predict(x_validation)
    metrics = calculate_metrics(y_validation, predictions)
    results.append({"model": name, **metrics})
    print(f"{name:28s} RMSE={metrics['rmse']:.2f}  MAE={metrics['mae']:.2f}  R2={metrics['r2']:.4f}")

results.sort(key=lambda result: (result["rmse"], result["mae"]))
winner_name = results[0]["model"]
print("Selected model:", winner_name)

Dummy Mean                   RMSE=90.76  MAE=69.05  R2=-0.0000


Linear Regression            RMSE=41.63  MAE=31.17  R2=0.7896
Ridge Regression             RMSE=41.63  MAE=31.17  R2=0.7896


Lasso Regression             RMSE=41.66  MAE=31.13  R2=0.7893


Elastic Net                  RMSE=41.60  MAE=31.05  R2=0.7899


Huber Regression             RMSE=41.83  MAE=30.92  R2=0.7876


...

Extra Trees                  RMSE=41.89  MAE=30.28  R2=0.7869


Gradient Boosting            RMSE=42.00  MAE=30.24  R2=0.7859


Histogram Gradient Boosting  RMSE=39.16  MAE=28.49  R2=0.8139


AdaBoost                     RMSE=53.24  MAE=41.46  R2=0.6559


XGBoost                      RMSE=39.45  MAE=28.59  R2=0.8111
Selected model: Histogram Gradient Boosting


## 5. Retrain and Evaluate the Winner

In [11]:
final_preprocessor = Pipeline([
    ("vectorizer", DictVectorizer(sparse=False)),
    ("scaler", StandardScaler()),
])

x_development = final_preprocessor.fit_transform(x_development_rows).astype(np.float32)
x_test = final_preprocessor.transform(x_test_rows).astype(np.float32)

if winner_name == "RBF SVR" and len(y_development) > 12000:
    random_generator = np.random.default_rng(RANDOM_STATE)
    selected_rows = random_generator.choice(len(y_development), 12000, replace=False)
    final_model = clone(models[winner_name]).fit(
        x_development[selected_rows], y_development[selected_rows]
    )
else:
    final_model = clone(models[winner_name]).fit(x_development, y_development)

### Evaluate once on untouched test data

In [12]:
test_predictions = final_model.predict(x_test)
final_metrics = calculate_metrics(y_test, test_predictions)

print(f"MAE:  INR {final_metrics['mae']:.2f}")
print(f"RMSE: INR {final_metrics['rmse']:.2f}")
print(f"R2:   {final_metrics['r2']:.4f}")

MAE:  INR 28.60
RMSE: INR 39.21
R2:   0.8320


## 6. Save the Runtime Artifacts

In [13]:
joblib.dump(final_model, MODELS_DIR / "budget_regressor.pkl")
joblib.dump(final_preprocessor, MODELS_DIR / "spending_preprocessor.pkl")

print("Saved the model and preprocessor in:", MODELS_DIR)

Saved the model and preprocessor in: D:\The Intelligent Culinary Matchmaker & Personalized Menu Agent\culinary_matchmaker\models


## 7. Test a Sample Prediction

Load the saved runtime artifacts and estimate one user's expected order value.

In [ ]:
saved_model = joblib.load(MODELS_DIR / "budget_regressor.pkl")
saved_preprocessor = joblib.load(MODELS_DIR / "spending_preprocessor.pkl")

sample_user = {
    "user_average_order_value": 240,
    "meal_time": "Dinner",
    "weekday_or_weekend": "Weekend",
    "location": "Bengaluru",
    "payment_method": "UPI",
    "previous_order_count": 12,
    "preferred_cuisine": "North Indian",
}

sample_features = saved_preprocessor.transform([sample_user])
expected_order_value = max(0, saved_model.predict(sample_features)[0])

print(f"Expected order value: INR {expected_order_value:.2f}")
print(f"Predicted spending limit: INR {round(expected_order_value)}")

## Interpretation

MAE and RMSE are measured in rupees because the target is an order amount. R² is unitless. The application uses the saved model to estimate a spending limit only when the user does not provide an explicit budget.